# 02 · Column profiling

**What this notebook decides:** it looks inside every column, grouped by what it means (ids, dates, amounts, codes, text, references), to decide its real type and what needs cleaning before it can be used. The result is the list of checks and transformations for silver (§3.9).

In [23]:
from helpers import connect, show, profile, by_year, values_by_year, DATA_DIR

con, years = connect()
print(f"years: {years[0]}-{years[-1]} ({len(years)} files)")
show("SELECT count(*) AS total_rows FROM s")

years: 2012-2026 (15 files)
┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│     506977 │
└────────────┘



## 3. Column profile
`profile("COLUMN")` shows, for one column:
- **blank**: exactly `''`
- **spaces_only**: only spaces, like `' '`
- **padded**: has spaces at the start or end
- **leading_zero**: starts with `0` and is longer than one character (the zero would be lost if converted to a number)
- **distinct** values and the **length** range
- **pct_int / pct_num / pct_date**: share of non-blank values that read as a whole number / number / `yyyy-mm-dd` date
- the most common values

**Question:** what does the whole table look like at a glance?

In [24]:
show("SUMMARIZE s", rows=30)

┌─────────────────────────┬─────────────┬────────────────────────────┬───────────────────────────────────┬───────────────┬────────────────────┬───────────────────┬─────────┬─────────┬─────────┬────────┬─────────────────┐
│       column_name       │ column_type │            min             │                max                │ approx_unique │        avg         │        std        │   q25   │   q50   │   q75   │ count  │ null_percentage │
│         varchar         │   varchar   │          varchar           │              varchar              │     int64     │      varchar       │      varchar      │ varchar │ varchar │ varchar │ int64  │  decimal(9,2)   │
├─────────────────────────┼─────────────┼────────────────────────────┼───────────────────────────────────┼───────────────┼────────────────────┼───────────────────┼─────────┼─────────┼─────────┼────────┼─────────────────┤
│ file_year               │ INTEGER     │ 2012                       │ 2026                              │          

### 3.0 Is every file in the same format?

**Question:** the data comes as 15 separate year files. Before applying one rule to all of them: do they have the same columns, and are values like dates written the same way in every file?

In [25]:
# Do all 15 bronze tables have the same columns, in the same order, with the same types?
show("""
SELECT count(*) AS tables,
       count(DISTINCT column_list) AS distinct_column_lists,
       count(DISTINCT type_list) AS distinct_type_lists,
       min(n_columns) AS min_columns, max(n_columns) AS max_columns,
       sum(not_varchar) AS columns_not_varchar
FROM (SELECT table_name, count(*) AS n_columns,
             count(*) FILTER (WHERE data_type <> 'VARCHAR') AS not_varchar,
             string_agg(column_name, ',' ORDER BY ordinal_position) AS column_list,
             string_agg(data_type, ',' ORDER BY ordinal_position) AS type_list
      FROM information_schema.columns
      WHERE table_schema = 'bronze' AND table_name IN (""" + ", ".join(f"'sales_{y}'" for y in years) + """)
      GROUP BY table_name)
""")

┌────────┬───────────────────────┬─────────────────────┬─────────────┬─────────────┬─────────────────────┐
│ tables │ distinct_column_lists │ distinct_type_lists │ min_columns │ max_columns │ columns_not_varchar │
│ int64  │         int64         │        int64        │    int64    │    int64    │       int128        │
├────────┼───────────────────────┼─────────────────────┼─────────────┼─────────────┼─────────────────────┤
│     15 │                     1 │                   1 │          26 │          26 │                   0 │
└────────┴───────────────────────┴─────────────────────┴─────────────┴─────────────┴─────────────────────┘



In [26]:
# Per year file: the shape of each date (digits replaced by 9), the largest middle and last date part,
# the lengths of the fixed-length columns, and prices that are not plain digits
show("""
SELECT file_year,
       list(DISTINCT regexp_replace(SALEDATE, '[0-9]', '9', 'g') ORDER BY regexp_replace(SALEDATE, '[0-9]', '9', 'g'))       AS saledate_shapes,
       list(DISTINCT regexp_replace(RECORDDATE, '[0-9]', '9', 'g') ORDER BY regexp_replace(RECORDDATE, '[0-9]', '9', 'g'))   AS recorddate_shapes,
       max(try_cast(split_part(SALEDATE, '-', 2) AS INT))   AS sale_middle_max,
       max(try_cast(split_part(SALEDATE, '-', 3) AS INT))   AS sale_last_max,
       max(try_cast(split_part(RECORDDATE, '-', 2) AS INT)) AS record_middle_max,
       max(try_cast(split_part(RECORDDATE, '-', 3) AS INT)) AS record_last_max,
       list(DISTINCT length(PARID) ORDER BY length(PARID))             AS parid_len,
       list(DISTINCT length(SCHOOLCODE) ORDER BY length(SCHOOLCODE))   AS school_len,
       list(DISTINCT length(MUNICODE) ORDER BY length(MUNICODE))       AS muni_len,
       list(DISTINCT length(PROPERTYZIP) ORDER BY length(PROPERTYZIP)) AS zip_len,
       count(*) FILTER (WHERE NOT regexp_matches(PRICE, '^[0-9]*$'))   AS price_not_digits
FROM s GROUP BY 1 ORDER BY 1
""", width=400)

┌───────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬───────────────────┬─────────────────┬───────────┬────────────┬──────────┬─────────┬──────────────────┐
│ file_year │ saledate_shapes │ recorddate_shapes │ sale_middle_max │ sale_last_max │ record_middle_max │ record_last_max │ parid_len │ school_len │ muni_len │ zip_len │ price_not_digits │
│   int32   │    varchar[]    │     varchar[]     │      int32      │     int32     │       int32       │      int32      │  int64[]  │  int64[]   │ int64[]  │ int64[] │      int64       │
├───────────┼─────────────────┼───────────────────┼─────────────────┼───────────────┼───────────────────┼─────────────────┼───────────┼────────────┼──────────┼─────────┼──────────────────┤
│      2012 │ [9999-99-99]    │ ['', 9999-99-99]  │              12 │            31 │                12 │              31 │ [16]      │ [2]        │ [3]      │ [5]     │                0 │
│      2013 │ [9999-99-99]    │ ['', 9999-99-99]  │    

**What this shows:** all 15 files have the same 26 columns in the same order, all text. Dates are written the same way everywhere: SALEDATE is always `9999-99-99`, and RECORDDATE is `9999-99-99` or blank. In every file the middle part of a date never goes above 12 while the last part goes up to 31 (2026 only runs to October, so its middle part stops at 10), so every file is year-month-day. PARID is always 16 characters, SCHOOLCODE 2, MUNICODE 3 and ZIP 5 (2017 also has one blank ZIP), and every price is digits or blank.

**So in silver:** one rule per column works for all 15 files (e.g. `%Y-%m-%d` for both dates). The source can change a file's format without notice, so the same shape checks run on every load: a file whose dates or lengths don't match is stopped, not loaded with a wrong reading. A *blank* value isn't a wrong shape: the one blank ZIP becomes NULL instead of stopping the load.

### 3.1 Ids

**Question:** is each id always filled, always the same format, and without extra spaces?

In [27]:
for col in ['_id', 'PARID']:
    profile(col)

┌─────────────┬───────┬─────────────┬────────┬──────────────┬─────────────────┬──────────────┬─────────┬─────────┬──────────┐
│ column_name │ blank │ spaces_only │ padded │ leading_zero │ distinct_values │ length_range │ pct_int │ pct_num │ pct_date │
│   varchar   │ int64 │    int64    │ int64  │    int64     │      int64      │   varchar    │ double  │ double  │  double  │
├─────────────┼───────┼─────────────┼────────┼──────────────┼─────────────────┼──────────────┼─────────┼─────────┼──────────┤
│ _id         │     0 │           0 │      0 │            0 │          484333 │ 7-8          │   100.0 │   100.0 │      0.0 │
└─────────────┴───────┴─────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────┴─────────┴──────────┘

┌──────────┬───────┬───────┐
│  value   │  len  │ rows  │
│ varchar  │ int64 │ int64 │
├──────────┼───────┼───────┤
│ 21317774 │     8 │     2 │
│ 21317917 │     8 │     2 │
│ 21317788 │     8 │     2 │
│ 21317977 │     8 │     2 │
│ 21317938 

In [28]:
# PARID shape: digits -> 9, letters -> A (anything else would show as itself)
show("""
SELECT regexp_replace(regexp_replace(PARID, '[0-9]', '9', 'g'), '[A-Z]', 'A', 'g') AS shape,
       count(*) AS rows, count(DISTINCT PARID) AS parcels, any_value(PARID) AS example
FROM s GROUP BY 1 ORDER BY rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────┬─────────┬──────────────────┐
│      shape       │  rows  │ parcels │     example      │
│     varchar      │ int64  │  int64  │     varchar      │
├──────────────────┼────────┼─────────┼──────────────────┤
│ 9999A99999999999 │ 498772 │  309096 │ 1075F00108000000 │
│ 9999A99999999A99 │   5870 │    3498 │ 0023M00208000A00 │
│ 9999A99999A99999 │   1529 │     852 │ 0829B00001F00700 │
│ 9999A9999999A999 │    281 │     176 │ 0025F0017700D300 │
│ 9999A999999A9999 │    162 │      88 │ 0111K002050O6700 │
│ 9999A99999AA9999 │     75 │      35 │ 0922L00001SE1000 │
│ 9999A9999999AA99 │     72 │      52 │ 0849R0002500MC00 │
│ 9999A999999AA999 │     59 │      24 │ 0012J002470RU100 │
│ 9999A999999A9A99 │     53 │      28 │ 0311K004000H1A00 │
│ 9999A99999A99A99 │     38 │      21 │ 0103A00075B15A00 │
│ 9999A99999AA9A99 │     36 │      17 │ 9910X00999AC6C00 │
│ 9999A99999AAA999 │     12 │       8 │ 0012J00247SHU400 │
│ 9999A999999A999A │      9 │       5 │ 0103E002000A120C

**What this shows:** PARID stays text and gets a fixed-length check (16 characters). `_id` is kept only to trace a cleaned row back to its bronze row.

### 3.2 Dates

**Question:** what format are the dates really in, and how many are missing?

In [29]:
for col in ['SALEDATE', 'RECORDDATE']:
    profile(col)

┌─────────────┬───────┬─────────────┬────────┬──────────────┬─────────────────┬──────────────┬─────────┬─────────┬──────────┐
│ column_name │ blank │ spaces_only │ padded │ leading_zero │ distinct_values │ length_range │ pct_int │ pct_num │ pct_date │
│   varchar   │ int64 │    int64    │ int64  │    int64     │      int64      │   varchar    │ double  │ double  │  double  │
├─────────────┼───────┼─────────────┼────────┼──────────────┼─────────────────┼──────────────┼─────────┼─────────┼──────────┤
│ SALEDATE    │     0 │           0 │      0 │            0 │            5202 │ 10-10        │     0.0 │     0.0 │    100.0 │
└─────────────┴───────┴─────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────┴─────────┴──────────┘

┌────────────┬───────┬───────┐
│   value    │  len  │ rows  │
│  varchar   │ int64 │ int64 │
├────────────┼───────┼───────┤
│ 2012-10-26 │    10 │   586 │
│ 2016-04-29 │    10 │   583 │
│ 2013-04-26 │    10 │   559 │
│ 2012-01-11 │    10 │   4

**Question:** which part of the date is the month? A value above 12 can only be a day, so the part that never goes above 12 while the other part does is the month.

In [30]:
for col in ["SALEDATE", "RECORDDATE"]:
    show(f"""
    SELECT '{col}' AS column_name,
           count(*) FILTER (WHERE trim({col}) = '') AS blank,
           count(*) FILTER (WHERE trim({col}) <> '' AND NOT regexp_matches({col}, '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$')) AS not_dddd_dd_dd,
           min(try_cast(split_part({col}, '-', 2) AS INT)) AS part2_min, max(try_cast(split_part({col}, '-', 2) AS INT)) AS part2_max,
           count(DISTINCT try_cast(split_part({col}, '-', 2) AS INT)) AS part2_distinct,
           min(try_cast(split_part({col}, '-', 3) AS INT)) AS part3_min, max(try_cast(split_part({col}, '-', 3) AS INT)) AS part3_max,
           count(DISTINCT try_cast(split_part({col}, '-', 3) AS INT)) AS part3_distinct,
           count(*) FILTER (WHERE try_cast(split_part({col}, '-', 3) AS INT) > 12) AS rows_part3_above_12
    FROM s
    """, width=300)

┌─────────────┬───────┬────────────────┬───────────┬───────────┬────────────────┬───────────┬───────────┬────────────────┬─────────────────────┐
│ column_name │ blank │ not_dddd_dd_dd │ part2_min │ part2_max │ part2_distinct │ part3_min │ part3_max │ part3_distinct │ rows_part3_above_12 │
│   varchar   │ int64 │     int64      │   int32   │   int32   │     int64      │   int32   │   int32   │     int64      │        int64        │
├─────────────┼───────┼────────────────┼───────────┼───────────┼────────────────┼───────────┼───────────┼────────────────┼─────────────────────┤
│ SALEDATE    │     0 │              0 │         1 │        12 │             12 │         1 │        31 │             31 │              317192 │
└─────────────┴───────┴────────────────┴───────────┴───────────┴────────────────┴───────────┴───────────┴────────────────┴─────────────────────┘

┌─────────────┬───────┬────────────────┬───────────┬───────────┬────────────────┬───────────┬───────────┬────────────────┬───────

**Question:** are the dates in the format the data dictionary says (`dd/mm/yyyy`)? How many record dates are blank?

In [31]:
show("""
SELECT count(*) FILTER (WHERE SALEDATE <> '' AND try_strptime(SALEDATE, '%Y-%m-%d') IS NULL)   AS sale_not_iso,
       count(*) FILTER (WHERE SALEDATE <> '' AND try_strptime(SALEDATE, '%d/%m/%Y') IS NOT NULL) AS sale_is_dd_mm_yyyy,
       count(*) FILTER (WHERE RECORDDATE = '')                                                 AS record_blank
FROM s
""")

┌──────────────┬────────────────────┬──────────────┐
│ sale_not_iso │ sale_is_dd_mm_yyyy │ record_blank │
│    int64     │       int64        │    int64     │
├──────────────┼────────────────────┼──────────────┤
│            0 │                  0 │         1249 │
└──────────────┴────────────────────┴──────────────┘



**What this shows:** the dates are year-month-day, not dd/mm/yyyy as the data dictionary says. Every value is dddd-dd-dd, and 0 sale dates are dd/mm/yyyy. The middle part never goes above 12 (12 values), while the last part goes up to 31 (31 values) and is above 12 in 317,192 sale dates and 303,301 record dates. A value above 12 can only be a day, so the middle part is the month.

SALEDATE is never blank. RECORDDATE is blank in 1,249 rows, and 1 record date starts with a zero, which can't be a real year.

**So:** both columns become DATE, read with `%Y-%m-%d`. A blank RECORDDATE becomes NULL, and a record date that isn't a real date is flagged.

**Open question:** the data dictionary says dd/mm/yyyy; can WPRDC confirm it's year-month-day?

**Question:** once converted, are all values real dates in a sensible range: none in the future, none before the data starts (2012)?

In [32]:
# Dates converted to DATE: blank, not a real date, before 2012, after today, first and last date
show("""
SELECT column_name,
       count(*) FILTER (WHERE trim(v) = '')                AS blank,
       count(*) FILTER (WHERE trim(v) <> '' AND d IS NULL) AS not_a_date,
       count(*) FILTER (WHERE d < DATE '2012-01-01')       AS before_2012,
       count(*) FILTER (WHERE d > current_date)            AS after_today,
       min(d) AS first_date, max(d) AS last_date
FROM (SELECT 'SALEDATE' AS column_name, SALEDATE AS v, try_strptime(SALEDATE, '%Y-%m-%d')::DATE AS d FROM s
      UNION ALL
      SELECT 'RECORDDATE', RECORDDATE, try_strptime(RECORDDATE, '%Y-%m-%d')::DATE FROM s)
GROUP BY 1 ORDER BY 1 DESC
""")
# The record dates before 2012
show("""
SELECT file_year, PARID, SALEDATE, RECORDDATE, PRICE, SALECODE, SALEDESC
FROM s WHERE try_strptime(RECORDDATE, '%Y-%m-%d') < DATE '2012-01-01'
ORDER BY RECORDDATE
""", width=260)

┌─────────────┬───────┬────────────┬─────────────┬─────────────┬────────────┬────────────┐
│ column_name │ blank │ not_a_date │ before_2012 │ after_today │ first_date │ last_date  │
│   varchar   │ int64 │   int64    │    int64    │    int64    │    date    │    date    │
├─────────────┼───────┼────────────┼─────────────┼─────────────┼────────────┼────────────┤
│ SALEDATE    │     0 │          0 │           0 │           0 │ 2012-01-01 │ 2026-10-22 │
│ RECORDDATE  │  1249 │          0 │           4 │           0 │ 0212-08-01 │ 2026-10-29 │
└─────────────┴───────┴────────────┴─────────────┴─────────────┴────────────┴────────────┘

┌───────────┬──────────────────┬────────────┬────────────┬─────────┬──────────┬─────────────────────────┐
│ file_year │      PARID       │  SALEDATE  │ RECORDDATE │  PRICE  │ SALECODE │        SALEDESC         │
│   int32   │     varchar      │  varchar   │  varchar   │ varchar │ varchar  │         varchar         │
├───────────┼──────────────────┼────────────

**What this shows:** SALEDATE is clean: every value is a real date between 2012-01-01 and 2026-10-22. RECORDDATE has 1,249 blanks, 0 impossible dates and 0 future dates, but 4 record dates are before 2012.

**So:** both columns become DATE. A record date before 2012 or after the file date is flagged, not corrected, because the right year could only be guessed. Sale dates are checked on every load so a future date can't get in.

**Open question:** are record dates 0212-08-01, 2006-04-11 and 2008-08-17 typos for 2012, 2016 and 2018?

### 3.3 Amounts

**Question:** is the price always a number? How many are blank, zero or very small?

In [33]:
for col in ['PRICE']:
    profile(col)

┌─────────────┬───────┬─────────────┬────────┬──────────────┬─────────────────┬──────────────┬─────────┬─────────┬──────────┐
│ column_name │ blank │ spaces_only │ padded │ leading_zero │ distinct_values │ length_range │ pct_int │ pct_num │ pct_date │
│   varchar   │ int64 │    int64    │ int64  │    int64     │      int64      │   varchar    │ double  │ double  │  double  │
├─────────────┼───────┼─────────────┼────────┼──────────────┼─────────────────┼──────────────┼─────────┼─────────┼──────────┤
│ PRICE       │  3108 │           0 │      0 │            0 │           39405 │ 0-9          │   100.0 │   100.0 │      0.0 │
└─────────────┴───────┴─────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────┴─────────┴──────────┘

┌─────────┬───────┬────────┐
│  value  │  len  │  rows  │
│ varchar │ int64 │ int64  │
├─────────┼───────┼────────┤
│ 1       │     1 │ 105558 │
│ 0       │     1 │  15813 │
│ 10      │     2 │   7332 │
│ 150000  │     6 │   3327 │
│         │

**What this shows:** PRICE is a whole number in every non-blank row (`pct_int` 100%) and blank in 3,108 rows. The most common prices are $1 (105,558 rows), $0 (15,813) and $10 (7,332). These are token amounts, not market prices, so the price alone doesn't say a row was a sale.

**So:** PRICE becomes an integer, and a blank becomes NULL (unknown); $0 stays 0. Whether a row is a market sale is decided by the sale code, not the price.

**Open question:** does a blank price mean something different from $0 (unknown vs. a transfer with no payment)?

**Question:** once converted to an integer, is every price valid (no negatives), and how are prices spread, from token amounts to the largest?

In [34]:
# PRICE converted to an integer: invalid values, range and distribution
show("""
SELECT count(*) FILTER (WHERE trim(PRICE) = '')                AS blank,
       count(*) FILTER (WHERE trim(PRICE) <> '' AND p IS NULL) AS not_a_number,
       count(*) FILTER (WHERE p < 0)                           AS negative,
       count(*) FILTER (WHERE p = 0)                           AS zero,
       count(*) FILTER (WHERE p BETWEEN 1 AND 999)             AS under_1000,
       count(*) FILTER (WHERE p >= 10000000)                   AS from_10m,
       count(*) FILTER (WHERE p >= 100000000)                  AS from_100m,
       max(p)                                                  AS max_price
FROM (SELECT PRICE, try_cast(PRICE AS BIGINT) AS p FROM s)
""")
show("""
SELECT unnest([1, 10, 25, 50, 75, 90, 99]) AS percentile,
       unnest(quantile_disc(p, [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])) AS price
FROM (SELECT try_cast(PRICE AS BIGINT) AS p FROM s)
""")

┌───────┬──────────────┬──────────┬───────┬────────────┬──────────┬───────────┬───────────┐
│ blank │ not_a_number │ negative │ zero  │ under_1000 │ from_10m │ from_100m │ max_price │
│ int64 │    int64     │  int64   │ int64 │   int64    │  int64   │   int64   │   int64   │
├───────┼──────────────┼──────────┼───────┼────────────┼──────────┼───────────┼───────────┤
│  3108 │            0 │        0 │ 15813 │     116566 │      402 │         2 │ 148752900 │
└───────┴──────────────┴──────────┴───────┴────────────┴──────────┴───────────┴───────────┘

┌────────────┬─────────┐
│ percentile │  price  │
│   int32    │  int64  │
├────────────┼─────────┤
│          1 │       0 │
│         10 │       1 │
│         25 │      10 │
│         50 │   93500 │
│         75 │  225000 │
│         90 │  400000 │
│         99 │ 1500000 │
└────────────┴─────────┘



**What this shows:** every non-blank price is a valid whole number, and none is negative. The spread is very uneven: a quarter of rows with a price are $10 or less (the 25th percentile is $10; the 3,108 blanks aren't counted), the median is $93,500, 90% are $400,000 or less and 99% are $1.5M or less. 116,566 rows are between $1 and $999, 402 rows are $10M or more, and 2 are over $100M, up to $148,752,900. So an average of raw prices means nothing: token transfers pull it down and a few huge ones pull it up.

**So:** PRICE becomes an integer. Any market analysis filters on the sale code (valid sales) and counts a multi-parcel deed's price once, instead of cutting prices by amount.

**Open question:** are the 402 prices of $10M or more real single sales, or deed totals and typing errors?

### 3.4 Codes

**Question:** which values does each code take? Do codes have leading zeros or extra spaces?

In [35]:
for col in ['SALECODE', 'SALEDESC', 'INSTRTYP', 'INSTRTYPDESC', 'SCHOOLCODE', 'SCHOOLDESC', 'MUNICODE', 'MUNIDESC']:
    profile(col)

┌─────────────┬───────┬─────────────┬────────┬──────────────┬─────────────────┬──────────────┬─────────┬─────────┬──────────┐
│ column_name │ blank │ spaces_only │ padded │ leading_zero │ distinct_values │ length_range │ pct_int │ pct_num │ pct_date │
│   varchar   │ int64 │    int64    │ int64  │    int64     │      int64      │   varchar    │ double  │ double  │  double  │
├─────────────┼───────┼─────────────┼────────┼──────────────┼─────────────────┼──────────────┼─────────┼─────────┼──────────┤
│ SALECODE    │     0 │           0 │      0 │            8 │              47 │ 1-2          │    75.8 │    75.8 │      0.0 │
└─────────────┴───────┴─────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────┴─────────┴──────────┘

┌─────────┬───────┬────────┐
│  value  │  len  │  rows  │
│ varchar │ int64 │ int64  │
├─────────┼───────┼────────┤
│ 3       │     1 │ 104703 │
│ 0       │     1 │ 100458 │
│ H       │     1 │  67492 │
│ 14      │     2 │  45338 │
│ 36      │

One profile per code or description column. `distinct_values` = how many different values exist, `leading_zero` = how many start with 0, `padded` = how many have spaces around them.

**What this shows:** all codes are filled (0 blanks), and the code columns have no extra spaces.
- **SALECODE has 47 values but SALEDESC only 27,** so some codes share a description: `OTHER` is used by the valid codes U and UR and by discontinued codes. Validity has to come from SALECODE using the dictionary; SALEDESC can't tell valid from invalid.
- **INSTRTYP and INSTRTYPDESC** have 31 values each.
- **SCHOOLCODE (89,206 rows) and SALECODE (8 rows) have leading zeros,** so codes have to stay text.
- **MUNIDESC has extra spaces in 360,151 rows,** so it has to be trimmed before it's shown.
- **INSTRTYPDESC mixes upper and mixed case** (Mobile Home Sale, Memorandum of Lease).

**Question:** does every code belong to a known list? SALECODE is checked against the code list in the data dictionary, and the other codes against their fixed format.

In [36]:
# SALECODE against the data dictionary: active codes, discontinued codes, and codes it doesn't list
active = ["AA", "0", "U", "UR", "1", "32", "2", "GV", "3", "6", "9", "13", "14", "16", "19", "27", "33", "34",
          "35", "36", "37", "99", "BK", "DT", "H", "N", "PA"]
discontinued = ["UX", "VB", "VH", "0X", "50", "PX", "X", "XX", "YY", "Z", "21", "24", "25", "26", "28", "29",
                "4", "5", "7", "8", "S", "V", "ND"]
in_list = lambda codes: ", ".join(f"'{c}'" for c in codes)
show(f"""
SELECT CASE WHEN SALECODE IN ({in_list(active)}) THEN 'active (in dictionary)'
            WHEN SALECODE IN ({in_list(discontinued)}) THEN 'discontinued (in dictionary)'
            ELSE 'not in dictionary' END AS dictionary_status,
       count(DISTINCT SALECODE) AS codes, count(*) AS rows,
       list(DISTINCT SALECODE ORDER BY SALECODE) AS code_values
FROM s GROUP BY 1 ORDER BY rows DESC
""", width=300)
show(f"""
SELECT SALECODE, SALEDESC, count(*) AS rows, min(file_year) AS first_year, max(file_year) AS last_year
FROM s WHERE SALECODE NOT IN ({in_list(active + discontinued)}) GROUP BY 1, 2
""", width=260)
# Fixed-format codes
show("""
SELECT count(*) FILTER (WHERE NOT regexp_matches(SCHOOLCODE, '^[0-9]{2}$')) AS schoolcode_not_2_digits,
       count(*) FILTER (WHERE NOT regexp_matches(MUNICODE, '^[0-9]{3}$'))   AS municode_not_3_digits,
       count(*) FILTER (WHERE NOT regexp_matches(INSTRTYP, '^[A-Z]{2}$'))   AS instrtyp_not_2_letters
FROM s
""")

┌──────────────────────────────┬───────┬────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────┐
│      dictionary_status       │ codes │  rows  │                                             code_values                                             │
│           varchar            │ int64 │ int64  │                                              varchar[]                                              │
├──────────────────────────────┼───────┼────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ active (in dictionary)       │    27 │ 499840 │ [0, 1, 13, 14, 16, 19, 2, 27, 3, 32, 33, 34, 35, 36, 37, 6, 9, 99, AA, BK, DT, GV, H, N, PA, U, UR] │
│ discontinued (in dictionary) │    19 │   7104 │ [0X, 21, 28, 29, 4, 5, 50, 8, ND, PX, S, UX, V, VB, VH, X, XX, YY, Z]                               │
│ not in dictionary            │     1 │     33 │ [PV]                                  

**What this shows:** 27 active codes cover 499,840 rows, and 19 discontinued codes cover 7,104 rows. One code isn't in the dictionary at all: `PV`, 33 rows from 2016 to 2025. SCHOOLCODE is always 2 digits, MUNICODE 3 digits and INSTRTYP 2 capital letters (0 exceptions).

**So:** a lookup table built from the data dictionary gives each sale code a group: valid (`0`, `U`, `UR`), undetermined (`AA`), or invalid (discontinued codes count as invalid, as the dictionary says). A code missing from the lookup gives a warning on load, and `PV` stays unclassified until the question below is answered.

**Open question:** what does sale code `PV` mean, and should it count as a valid or invalid sale?

### 3.5 Address

**Question:** which address parts are filled, have extra spaces, or are placeholders?

In [37]:
for col in ['FULL_ADDRESS', 'PROPERTYHOUSENUM', 'PROPERTYFRACTION', 'PROPERTYADDRESSDIR', 'PROPERTYADDRESSSTREET', 'PROPERTYADDRESSSUF', 'PROPERTYADDRESSUNITDESC', 'PROPERTYUNITNO', 'PROPERTYCITY', 'PROPERTYSTATE', 'PROPERTYZIP']:
    profile(col)

┌──────────────┬───────┬─────────────┬────────┬──────────────┬─────────────────┬──────────────┬─────────┬─────────┬──────────┐
│ column_name  │ blank │ spaces_only │ padded │ leading_zero │ distinct_values │ length_range │ pct_int │ pct_num │ pct_date │
│   varchar    │ int64 │    int64    │ int64  │    int64     │      int64      │   varchar    │ double  │ double  │  double  │
├──────────────┼───────┼─────────────┼────────┼──────────────┼─────────────────┼──────────────┼─────────┼─────────┼──────────┤
│ FULL_ADDRESS │     0 │           0 │      0 │        40131 │          288370 │ 15-67        │     0.0 │     0.0 │      0.0 │
└──────────────┴───────┴─────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────┴─────────┴──────────┘

┌───────────────────────────────────────────┬───────┬───────┐
│                   value                   │  len  │ rows  │
│                  varchar                  │ int64 │ int64 │
├───────────────────────────────────────────┼──────

**What this shows:**
- PROPERTYSTATE is always PA. 40,131 rows have house number 0, and exactly as many full addresses start with 0. The house number is blank in another 7 rows.
- PROPERTYFRACTION is a single space in 494,892 rows and has extra spaces in 12,085. Direction, unit description and unit number are blank in most rows (485,929, 494,631 and 495,015), probably because they're optional parts of an address. A few rows have no street, city or ZIP, and city and street have extra spaces.

**So:** address parts are trimmed, a value of only spaces becomes NULL, house number 0 is treated as unknown, and PROPERTYSTATE can be dropped or kept as a constant.

**Open question:** does house number 0 mean a lot with no number, or a missing number?

**Question:** do the address parts follow their expected format: 5-digit ZIP codes in the county's range, house numbers made of digits, and directions and unit types from a small known list?

In [38]:
# Address parts against their expected format and allowed values
show("""
SELECT count(*) FILTER (WHERE trim(PROPERTYZIP) <> '' AND NOT regexp_matches(PROPERTYZIP, '^[0-9]{5}$')) AS zip_not_5_digits,
       count(*) FILTER (WHERE trim(PROPERTYHOUSENUM) <> '' AND NOT regexp_matches(PROPERTYHOUSENUM, '^[0-9]+$')) AS housenum_not_digits,
       count(*) FILTER (WHERE PROPERTYSTATE <> 'PA') AS state_not_pa,
       list(DISTINCT trim(PROPERTYADDRESSDIR) ORDER BY trim(PROPERTYADDRESSDIR))           AS direction_values,
       list(DISTINCT trim(PROPERTYADDRESSUNITDESC) ORDER BY trim(PROPERTYADDRESSUNITDESC)) AS unit_desc_values
FROM s
""", width=300)
# ZIP codes outside 150xx-152xx: which places and municipalities?
show("""
SELECT PROPERTYZIP, PROPERTYCITY, count(*) AS rows,
       list(DISTINCT trim(MUNIDESC) ORDER BY trim(MUNIDESC)) AS municipalities
FROM s WHERE regexp_matches(PROPERTYZIP, '^[0-9]{5}$') AND PROPERTYZIP NOT BETWEEN '15000' AND '15299'
GROUP BY 1, 2 ORDER BY rows DESC
""", width=300)

┌──────────────────┬─────────────────────┬──────────────┬──────────────────┬────────────────────────────────────────────────────────┐
│ zip_not_5_digits │ housenum_not_digits │ state_not_pa │ direction_values │                    unit_desc_values                    │
│      int64       │        int64        │    int64     │    varchar[]     │                       varchar[]                        │
├──────────────────┼─────────────────────┼──────────────┼──────────────────┼────────────────────────────────────────────────────────┤
│                0 │                   0 │            0 │ ['', E, N, S, W] │ ['', APT, BLDG, FRNT, GAR, LOT, REAR, STE, TRLR, UNIT] │
└──────────────────┴─────────────────────┴──────────────┴──────────────────┴────────────────────────────────────────────────────────┘

┌─────────────┬──────────────┬───────┬───────────────────────────────┐
│ PROPERTYZIP │ PROPERTYCITY │ rows  │        municipalities         │
│   varchar   │   varchar    │ int64 │           varc

The first table counts ZIP codes that aren't 5 digits, house numbers with non-digits and states other than PA, and lists every direction and unit description used. The second shows the ZIP codes outside 150xx–152xx with their city and municipality.

**What this shows:** every non-blank ZIP is 5 digits (1 is blank), every house number is digits only, the state is always PA, directions are only '', N, S, E and W, and unit descriptions come from 9 values (APT, BLDG, FRNT, GAR, LOT, REAR, STE, TRLR, UNIT).

The second table shows 3,165 rows with a ZIP outside 150xx–152xx. A "county range" check on ZIP would wrongly flag these.

**Open question:** is there an official list of ZIP codes or municipalities to check these addresses against?

### 3.6 Deed references

**Question:** what do deed book and page values look like? Placeholders, prefixes, leading zeros?

In [39]:
for col in ['DEEDBOOK', 'DEEDPAGE']:
    profile(col)

┌─────────────┬───────┬─────────────┬────────┬──────────────┬─────────────────┬──────────────┬─────────┬─────────┬──────────┐
│ column_name │ blank │ spaces_only │ padded │ leading_zero │ distinct_values │ length_range │ pct_int │ pct_num │ pct_date │
│   varchar   │ int64 │    int64    │ int64  │    int64     │      int64      │   varchar    │ double  │ double  │  double  │
├─────────────┼───────┼─────────────┼────────┼──────────────┼─────────────────┼──────────────┼─────────┼─────────┼──────────┤
│ DEEDBOOK    │   567 │           0 │      4 │         1286 │            6180 │ 0-8          │    99.0 │    99.0 │      0.0 │
└─────────────┴───────┴─────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────┴─────────┴──────────┘

┌─────────┬───────┬───────┐
│  value  │  len  │ rows  │
│ varchar │ int64 │ int64 │
├─────────┼───────┼───────┤
│ TR18    │     4 │  1232 │
│ 0       │     1 │  1114 │
│ TR13    │     4 │   938 │
│ 00      │     2 │   824 │
│ TR16    │     4 │

**What this shows:**
- DEEDBOOK is blank in 567 rows and DEEDPAGE in 579. Placeholders like 0, 00 and 000 are among the most common deed books, and books starting with TR are common too, so the book isn't always a number (99% read as a whole number).
- DEEDPAGE has leading zeros in 3,452 rows. A few values have extra spaces.

**So:** book and page stay text and are trimmed, and blank or zero placeholders become NULL. Leading zeros aren't removed until the question below is answered.

**Open question:** are deed page 00002 and 2 the same page?

### 3.7 Changes across years

**Question:** are blanks, extra spaces and leading zeros the same every year, or concentrated in some years? (rows per column per file)

In [40]:
by_year("blank")

┌─────────────────────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│       column_name       │  2012  │  2013  │  2014  │  2015  │  2016  │  2017  │  2018  │  2019  │  2020  │  2021  │  2022  │  2023  │  2024  │  2025  │  2026  │
│         varchar         │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │
├─────────────────────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ DEEDBOOK                │     39 │    178 │     41 │     60 │     16 │     12 │     32 │     41 │     63 │     72 │      8 │      1 │      4 │      0 │      0 │
│ DEEDPAGE                │     38 │    183 │     44 │     59 │     16 │     12 │     39 │     41 │     62 │     72 │      7 │      1 │      5 │      0 │      0 │
│ FULL_ADDRESS        

In [41]:
by_year("padded")

┌─────────────────────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│       column_name       │  2012  │  2013  │  2014  │  2015  │  2016  │  2017  │  2018  │  2019  │  2020  │  2021  │  2022  │  2023  │  2024  │  2025  │  2026  │
│         varchar         │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │
├─────────────────────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ DEEDBOOK                │      4 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │
│ DEEDPAGE                │      8 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │      0 │
│ FULL_ADDRESS        

In [42]:
by_year("leading_zero")

┌─────────────────────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│       column_name       │  2012  │  2013  │  2014  │  2015  │  2016  │  2017  │  2018  │  2019  │  2020  │  2021  │  2022  │  2023  │  2024  │  2025  │  2026  │
│         varchar         │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │ int128 │
├─────────────────────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ DEEDBOOK                │     48 │     66 │     41 │     58 │     80 │     94 │    120 │    101 │    119 │    122 │    102 │     77 │    108 │     87 │     63 │
│ DEEDPAGE                │    135 │    208 │    222 │    704 │    643 │    203 │    224 │    216 │    209 │    122 │    103 │    120 │    134 │    145 │     64 │
│ FULL_ADDRESS        

**Question:** do the code values change over time? (most common values per file)

In [43]:
for col in ["SALECODE", "INSTRTYP"]:
    values_by_year(col)

┌─────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│  value  │ 2012  │ 2013  │ 2014  │ 2015  │ 2016  │ 2017  │ 2018  │ 2019  │ 2020  │ 2021  │ 2022  │ 2023  │ 2024  │ 2025  │ 2026  │
│ varchar │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │
├─────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┤
│ 0       │  2372 │  7420 │  3317 │  3784 │  5695 │  5843 │  4858 │  3014 │  8222 │  8831 │ 11011 │  9941 │  9607 │  9529 │  7014 │
│ 1       │  1469 │  1787 │  1440 │  1447 │  1528 │  1242 │  1336 │  1069 │   476 │   468 │   579 │   582 │   618 │   589 │   504 │
│ 14      │     6 │    12 │  2708 │  3075 │  2586 │  3042 │  3037 │  3203 │  5125 │  5191 │  4303 │  3590 │  3459 │  3531 │  2470 │
│ 16      │   137 │    38 │   504 │   540 │   512 │   501 │   473 │   412 │ 

The three `by_year` tables have one row per column and one column per year file; each cell is how many rows of that year have the problem (blank, extra spaces, leading zero). The last two tables show the most common sale codes and instrument types, with their row count in each year.

**What this shows:** DEEDBOOK and DEEDPAGE blanks are mostly in 2012 to 2021, and there are none in 2025 and 2026. RECORDDATE blanks are almost all in 2012 to 2016; later years have 16 or fewer. PRICE blanks drop from 218–381 a year before 2022 to 66–124 after. MUNIDESC has extra spaces every year, DEEDBOOK and DEEDPAGE only in 2012.

The sale coding seems to have changed over the years:
- Z is only used in 2012 and 2013.
- UR is only used in 2013 to 2019 (1,560 to 2,296 a year in 2014–2019, then 0 to 2 from 2020).
- Code 36 drops from 4,548–6,930 a year in 2013–2019 to 119–211 from 2020.
- Code 9 peaks in 2019–2021 (4,377, 5,401, 7,164).
- Code 0 (VALID SALE) rises from 2,372 in 2012 to 11,011 in 2022.
- AA (not analysed) is 9,903 in 2012, almost 0 in 2020–2025, and back to 1,217 in 2026.
- Instrument type CT only appears in 2013.

**So:** the number of valid-coded rows per year shows how sales were coded, not only the market, and the newest year still has many sales waiting for review. Counts of valid sales can't be compared across years without a mapping.

**Open question:** did the County change its sale coding over the years, and how do the old codes map to the new ones?

### 3.8 Checked and clean

**Question:** which columns have unusual characters, only one value, or a fixed length? (all columns, so nothing is skipped)

In [44]:
cols = [c for c in con.sql("SELECT * FROM s LIMIT 0").columns if c != "file_year"]
show(" UNION ALL ".join(
    f"SELECT '{c}' AS column_name, "
    f"count(*) FILTER (WHERE regexp_matches(\"{c}\", '[^\\x20-\\x7E]')) AS non_printable_or_non_ascii, "
    f"count(DISTINCT \"{c}\") AS distinct_values, "
    f"min(length(\"{c}\")) AS min_len, max(length(\"{c}\")) AS max_len FROM s"
    for c in cols), rows=30)

┌─────────────────────────┬────────────────────────────┬─────────────────┬─────────┬─────────┐
│       column_name       │ non_printable_or_non_ascii │ distinct_values │ min_len │ max_len │
│         varchar         │           int64            │      int64      │  int64  │  int64  │
├─────────────────────────┼────────────────────────────┼─────────────────┼─────────┼─────────┤
│ _id                     │                          0 │          484333 │       7 │       8 │
│ PARID                   │                          0 │          313906 │      16 │      16 │
│ FULL_ADDRESS            │                          0 │          288370 │      15 │      67 │
│ PROPERTYHOUSENUM        │                          0 │           10074 │       0 │       5 │
│ PROPERTYFRACTION        │                          0 │            2849 │       1 │       7 │
│ PROPERTYADDRESSDIR      │                          0 │               5 │       0 │       1 │
│ PROPERTYADDRESSSTREET   │                       

**What this shows:** no column has non-printable or non-ASCII characters. PARID (16), SCHOOLCODE (2), INSTRTYP (2), MUNICODE (3) and SALEDATE (10) always have the same length, and PROPERTYSTATE has only 1 value.

### 3.9 Before silver: checks, cleaning and transformations

Based on everything above, this is how I decided what a check does when it fails:
- **hard**: the build stops and the previous silver stays in place. Only for things that mean the file itself changed (renamed columns, most dates in a new format) or that my own code is wrong.
- **reject**: a single row that can't be read (bad parcel id, sale date, price or record date) goes to `silver.rejected_records` with the reason, and the build goes on. If more than 0.1% of a year's rows are rejected, that's a format change, so the build stops instead.
- **warn**: something changed that a person should look at; the build goes on.
- **info**: known problems in the source; counted on every build, and the rows stay in silver with a flag. Nothing is dropped.

**Structural checks**
| Check | Result now | On build |
|---|---|---|
| Same 26 columns in the same order in every file | true | hard |
| Silver rows + rejected rows = bronze rows, per year | true | hard |
| Bronze rows = the row count recorded when the version was loaded | true | hard |
| Row count equals the count the source reports | true | done in bronze: the download is refused and retried, so silver doesn't repeat it |
| Share of rejected rows per year above 0.1% | 0 rejected | hard |

**Per column**
| Column | Silver type | Cleaning | Transformation | Check on build |
|---|---|---|---|---|
| `PARID` | text | none | none | reject the row: blank, or not 16 digits/capitals |
| `SALEDATE` | DATE | none | read as `%Y-%m-%d` | reject the row: not a date; warn: outside its file's year |
| `RECORDDATE` | DATE | blank → NULL | read as `%Y-%m-%d` | reject the row: not a date; info: before 2012, after the file date, or before the sale date |
| `PRICE` | integer | blank → NULL | convert to integer | reject the row: not a whole number; info: $10M or more |
| `SALECODE` | text | none | validity group from `pipeline/ref/sale_validation_codes.csv` (`0`/`U`/`UR` valid, `AA` undetermined, discontinued codes invalid, `PV` unclassified) | warn: code not in the lookup |
| `SALEDESC`, `INSTRTYPDESC`, `SCHOOLDESC`, `MUNIDESC` | text | trim | none | none |
| `INSTRTYP` / `SCHOOLCODE` / `MUNICODE` | text | none | none | warn: not 2 letters / 2 digits / 3 digits |
| `DEEDBOOK` / `DEEDPAGE` | text | trim; blank/zero placeholders → NULL; leading zeros kept | `is_deed_placeholder`; one row per real deed in `silver.deed_reference` | info: placeholder rows |
| `PROPERTYHOUSENUM` | text | blank → NULL; `0` kept as written | `has_house_number` is false for `0` | warn: non-digits |
| `PROPERTYFRACTION`, address parts, `PROPERTYCITY` | text | trim; only spaces → NULL | none | none |
| `PROPERTYZIP` | text | blank → NULL | none | warn: not 5 digits |
| `PROPERTYSTATE` | text | none | none | none |

**Row-level rules**
| Rule |
|---|
| `PARID + DEEDBOOK + DEEDPAGE + SALEDATE` matches records but isn't unique: 90 groups repeat. Both rows are kept and flagged (`is_repeated_parcel_deed_date`); which one is right stays open until the County answers. No repeated group has two valid-coded rows |
| Blank/zero deed references are flagged and never used to group or join |
| A deed's price is the total for all its parcels, except treasurer deeds, so gold only uses deeds with one parcel |
| Market analysis only uses valid sale codes; sale codes aren't compared across years without a mapping |
| `_id` + file year kept only for tracing a row back |